# Classify Rollouts - Premise and Conclusion Extraction

This notebook uses OpenAI's structured outputs to classify model responses by extracting premises, conclusions, and evaluating logical relationships.

In [ ]:
import os
import json
from typing import Dict, Any, List, Optional
from openai import OpenAI
from dotenv import load_dotenv

CACHE_DIR = 'cache/experiments'

In [ ]:
def classify_premises_and_conclusion(response: str) -> Dict[str, Any]:
    """
    Classify a model response by extracting premises, conclusion, and evaluating logical relationships.
    
    Args:
        response: The model's response text to analyze
        
    Returns:
        Dict containing:
        - premises: List of premise statements
        - conclusion: The conclusion statement
        - premise_truth_values: List of booleans for each premise's truth value
        - follows_from_premises: Whether conclusion follows from premises (assuming premises are true)
        - refuse: Whether the model was unable to analyze the response
    """
    # Load environment variables and API key
    load_dotenv()
    api_key = os.getenv("OPENAI_API_KEY")
    
    if not api_key:
        raise ValueError("OPENAI_API_KEY not found in environment variables")
    
    # Initialize OpenAI client
    client = OpenAI(api_key=api_key)
    
    # Define the JSON schema for structured output
    schema = {
        "type": "object",
        "properties": {
            "premises": {
                "type": "array",
                "items": {"type": "string"},
                "description": "List of premise statements extracted from the response"
            },
            "conclusion": {
                "type": "string",
                "description": "The conclusion statement extracted from the response"
            },
            "premise_truth_values": {
                "type": "array",
                "items": {"type": "boolean"},
                "description": "Truth value for each premise (true if factually correct, false otherwise)"
            },
            "follows_from_premises": {
                "type": "boolean",
                "description": "Whether the conclusion logically follows from the premises, assuming the premises are true"
            },
            "refuse": {
                "type": "boolean",
                "description": "Set to true if unable to analyze the response"
            }
        },
        "required": ["premises", "conclusion", "premise_truth_values", "follows_from_premises", "refuse"],
        "additionalProperties": False
    }
    
    # Create the prompt
    system_prompt = """You are a logical analysis assistant. Your task is to analyze text and extract logical components.
    
For the given text, you must:
1. Extract all premises (statements used as evidence or assumptions)
2. Identify the conclusion (the main claim or result)
3. Evaluate whether each premise is factually true or false
4. Determine if the conclusion logically follows from the premises (assuming the premises are true)
5. Set refuse to true ONLY if you cannot complete the analysis (e.g., text is unintelligible or not logical reasoning)

Important:
- Premises should be distinct, atomic statements
- The conclusion should be the final claim or answer
- For 'follows_from_premises': evaluate pure logical validity assuming premises are true, regardless of their actual truth
- Set refuse to false for normal analysis, true only if unable to analyze"""
    
    user_prompt = f"""Analyze the following response and extract its logical structure:

{response}

Extract the premises, conclusion, evaluate truth values, and determine logical validity."""
    
    try:
        # Make the API call with structured output using GPT-5 API format
        resp = client.responses.create(
            model="gpt-5",
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "logical_analysis",
                    "schema": schema,
                    "strict": True
                }
            },
            temperature=0.0,  # Use low temperature for consistent analysis
            max_output_tokens=1000
        )
        
        # Parse and return the structured response
        result = json.loads(resp.output_text)
        return result
        
    except Exception as e:
        print(f"Error calling OpenAI API: {e}")
        # Return a refuse response on error
        return {
            "premises": [],
            "conclusion": "",
            "premise_truth_values": [],
            "follows_from_premises": False,
            "refuse": True
        }

## Example Usage

In [ ]:
# Example: Test the classification function
example_response = """
Let me analyze this step by step.

First, all birds have wings. Second, penguins are birds. 
Third, things with wings can typically fly.

Therefore, penguins can fly.
"""

result = classify_premises_and_conclusion(example_response)
print(json.dumps(result, indent=2))

## Load and Process Experiment Results

In [ ]:
import pickle
from pathlib import Path

def load_experiment_responses(model_name: str, dataset_name: str, split: str = "test") -> List[Dict[str, Any]]:
    """
    Load model responses from experiment cache.
    
    Args:
        model_name: Name of the model (e.g., 'google_gemma-2-2b-it')
        dataset_name: Name of the dataset (e.g., 'logical_deduction')
        split: 'train' or 'test'
        
    Returns:
        List of response dictionaries
    """
    cache_path = Path(CACHE_DIR) / model_name / dataset_name
    
    # Find the split directory
    split_dirs = list(cache_path.glob("split_*"))
    if not split_dirs:
        print(f"No split directories found for {model_name}/{dataset_name}")
        return []
    
    # Use the first split directory
    split_dir = split_dirs[0]
    
    # Find experiment directories
    exp_dirs = list(split_dir.glob("*"))
    if not exp_dirs:
        print(f"No experiment directories found in {split_dir}")
        return []
    
    # Use the first experiment directory
    exp_dir = exp_dirs[0]
    
    # Load the generations file
    gen_file = exp_dir / "data" / f"{split}_generations.pkl"
    if not gen_file.exists():
        print(f"Generations file not found: {gen_file}")
        return []
    
    with open(gen_file, 'rb') as f:
        data = pickle.load(f)
    
    return data

# Example: Load responses
# responses = load_experiment_responses("google_gemma-2-2b-it", "logical_deduction")
# print(f"Loaded {len(responses)} responses")

In [ ]:
def batch_classify_responses(responses: List[Dict[str, Any]], max_samples: Optional[int] = None) -> List[Dict[str, Any]]:
    """
    Classify multiple responses in batch.
    
    Args:
        responses: List of response dictionaries from experiments
        max_samples: Maximum number of samples to process (None for all)
        
    Returns:
        List of classification results
    """
    results = []
    
    # Limit samples if specified
    if max_samples:
        responses = responses[:max_samples]
    
    for i, response_data in enumerate(responses):
        print(f"Processing response {i+1}/{len(responses)}...")
        
        # Extract the response text (adjust key based on actual data structure)
        if isinstance(response_data, dict):
            response_text = response_data.get('response', '') or response_data.get('text', '')
        else:
            response_text = str(response_data)
        
        # Classify the response
        classification = classify_premises_and_conclusion(response_text)
        
        # Add metadata
        classification['response_index'] = i
        classification['original_response'] = response_text
        
        results.append(classification)
    
    return results

# Example usage:
# classifications = batch_classify_responses(responses, max_samples=5)
# for cls in classifications:
#     print(f"\nResponse {cls['response_index']}:")
#     print(f"  Premises: {cls['premises']}")
#     print(f"  Conclusion: {cls['conclusion']}")
#     print(f"  Follows from premises: {cls['follows_from_premises']}")

## Save Results

In [ ]:
def save_classifications(classifications: List[Dict[str, Any]], output_path: str):
    """
    Save classification results to JSON file.
    
    Args:
        classifications: List of classification results
        output_path: Path to save the JSON file
    """
    with open(output_path, 'w') as f:
        json.dump(classifications, f, indent=2)
    print(f"Saved {len(classifications)} classifications to {output_path}")

# Example:
# save_classifications(classifications, "classification_results.json")